In [1]:
import pandas as pd         # for data manipulation and analysis
import numpy as np      # for numerical operations
import emoji    # for handling emojis in text data
import re       # for regular expressions to clean text data

In [2]:
# Load the dataset

df=pd.read_csv('../data/cleaned_data.csv')

In [3]:
print("Total number of reviews:", len(df))

Total number of reviews: 379732


In [4]:
# Creating new features to identify potential fake reviews and duplicates

# word_count => counts the number of words in the review text
# apply => applies a function to each element in the 'review_text' column, where the function splits the text into words and counts them
# str(x) => ensures that the review text is treated as a string, even if there are missing values or non-string entries
# split() => splits the review text into a list of words, and len() counts the number of words in that list
# lambda function => an anonymous function used to perform the word count operation on each review text
df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))

In [5]:
# is_fake => identifies reviews that have a word count of 2 or less and a rating of 5, which may indicate potential fake reviews
# astype(int) => converts the boolean result of the condition into an integer (1 for True, 0 for False) to create a binary feature because fake reviews often have very short text and high ratings
df['is_fake'] = ((df['word_count'] <= 2) & (df['rating'] == 5)).astype(int)

In [6]:
# is_duplicate => identifies duplicate reviews based on the 'review_text' column, marking them as 1 if they are duplicates and 0 otherwise
# subset=['review_text'] => specifies that duplicates should be identified based on the 'review_text' column
# keep=False => marks all duplicates as True, meaning that if a review is duplicated, all instances of that review will be marked as duplicates
df['is_duplicate'] = df.duplicated(subset=['review_text'], keep=False).astype(int)

In [7]:
print("Total word count statistics:", df['word_count'].sum())
print("doubtful reviews count:", df['is_fake'].sum())
print("duplicate reviews count:", df['is_duplicate'].sum())

Total word count statistics: 681574
doubtful reviews count: 194554
duplicate reviews count: 379726


In [8]:
# filtering out reviews that are too short (length <= 1) as they may not provide meaningful information for sentiment analysis
# keep = 'first' => keeps the first occurrence of each duplicate review and marks subsequent duplicates as False, effectively removing duplicates while retaining one instance of each unique review
df_unique = df.drop_duplicates(subset=['review_text'], keep='first')

# copying the unique reviews dataframe to a new dataframe df_final for further analysis or modeling
df_final = df_unique.copy()

In [9]:
manual_reviews = [
    {'review_text': 'Amazing product! 😍 love it.', 'rating': 5},
    {'review_text': 'Best quality and fast delivery 🚀🔥', 'rating': 5},
    {'review_text': 'Value for money, highly recommended! 😊👍', 'rating': 5},
    {'review_text': 'The taste is just wow! 😋✨', 'rating': 5},
    {'review_text': 'Superb experience, will buy again! ❤️', 'rating': 5},
    {'review_text': 'Very bad experience 😡 strictly avoid.', 'rating': 1},
    {'review_text': 'Worst quality, waste of money 🤮👎', 'rating': 1},
    {'review_text': 'Broken item received, very disappointed 😫💢', 'rating': 1},
    {'review_text': 'Packaging was terrible and smell is bad 😷', 'rating': 2},
    {'review_text': 'Does not work at all! 💀🚫', 'rating': 1},
    {'review_text': 'It is okay, nothing special 😐', 'rating': 3},
    {'review_text': 'Average product, can be better 😕', 'rating': 3},
    {'review_text': 'Delivery was late but product is fine 🚚⌛', 'rating': 3},
    {'review_text': 'Good but expensive 💸🤔', 'rating': 3},
    {'review_text': 'Just an ordinary item 🤷‍♂️', 'rating': 3},
    {'review_text': 'Not worth the price, very disappointed 😞💔', 'rating': 2},
    {'review_text': 'Excellent quality and great value for money! 😍💰', 'rating': 5},
    {'review_text': 'Terrible customer service, will never buy again! 😡👎', 'rating': 1},
    {'review_text': 'The product is okay, but the delivery was late 🚚⌛', 'rating': 3},
    {'review_text': 'I love this product! It works perfectly and the quality is amazing! 😍👍', 'rating': 5},
    {'review_text': 'Worst purchase ever! The item arrived broken and the customer service was terrible. 😡👎', 'rating': 1},
    {'review_text': 'The product is decent, but the delivery was delayed by a week. 🚚⌛', 'rating': 3},
    {'review_text': 'Not worth the money. The quality is poor and it stopped working after a week. 😞💔', 'rating': 2},
    {'review_text': 'Amazing value for money! The product exceeded my expectations and the delivery was fast. 😍💰🚚', 'rating': 5},
    {'review_text': 'Terrible experience! The item was defective and the customer service was unhelpful. 😡👎', 'rating': 1},
    {'review_text': 'The product is average, but the delivery was on time. 🚚⌛', 'rating': 3},
    {"review_text": "Product ekdam bhari aahe 😍 loved it!", "rating": 5},
    {"review_text": "Ekdam bakwas quality, paise fukat gele 😡👎", "rating": 1},
    {"review_text": "Paisa wasool product! Mast aahe. ❤️", "rating": 5},
    {"review_text": "Quality khupach ghan aahe, don't buy 🤮", "rating": 1},
    {"review_text": "Product accha nahi hai 😡", "rating": 2},
    {"review_text": "खूप छान प्रॉडक्ट आहे, आवडलं! ✨", "rating": 5},
    {"review_text": "खूपच खराब प्रॉडक्ट, पैसे वाया गेले! 👎", "rating": 1},
    {"review_text": "पैसे वाया गेले, घेऊ नका. 👎", "rating": 1},
    {"review_text":"nice product, value for money! 😍", "rating": 5},
    {"review_text":"बकवास प्रॉडक्ट, वापरून बघा नका! 😡", "rating": 1},
    {"review_text":"मस्त प्रॉडक्ट आहे, आवडला! ❤️", "rating": 5},
    {"review_text": "खूप छान product आहे, value for money!", "rating": 5}, 
    {"review_text": "बकवास प्रॉडक्ट, वापरून बघू नका! Very bad quality.", "rating": 1}, 
    {"review_text": "Bahut badhiya product hai, delivery fast hoti.", "rating": 5}, 
    {"review_text": "मस्त आहे, quality is really superior, नक्की खरेदी करा.", "rating": 5}, 
    {"review_text": "Worst experience, काम नाही करत हे नीट, don't buy.", "rating": 1}, 
    {"review_text": "Average product, किंमत थोडी जास्त आहे (price is high).", "rating": 3}, 
    {"review_text": "बढ़िया काम कर रहा है, highly recommended!", "rating": 5},
    {"review_text": "Don't waste your money, पैसे वाया जातील तुमचे, नका घेऊ.", "rating": 1},
    {"review_text":"product accha nahi hai 😡", "rating": 2},
    {"review_text": "खूपच भारी product आहे, value for money!", "rating": 5},
    {"review_text": "बढ़िया काम कर रहा है, highly recommended for everyone.", "rating": 5},
    {"review_text": "Quality is best, नक्की खरेदी करा, आवडलं मला.", "rating": 5},
    {"review_text": "Superb delivery! काल ऑर्डर केलं आणि आज मिळालं, fast service.", "rating": 5},
    {"review_text": "मस्त आहे, colors are very vibrant, go for it!", "rating": 4},
    {"review_text":"product quality accha hai, paisa vasool! 😍", "rating": 5},
    {"review_text":"product mast hai,but delivery late hui 😡", "rating": 3},
    {"review_text":"product okk hai, but packaging was bad 😕", "rating": 3},
    {"review_text":"nice product, but color was different than shown 😕", "rating": 4},
    {"review_text":"product quality accha hai, paisa vasool! 😍", "rating": 5},
    {"review_text":"product okk hai, but delivery was slow 😕", "rating": 3},
    {'review_text': 'I am very satisfied with this purchase! The product is of great quality and the delivery was prompt. 😍👍🚚', 'rating': 5},
    {'review_text': 'The worst product I have ever bought! It stopped working after a day and the customer service was terrible. 😡👎', 'rating': 1},
    {'review_text': 'The product is okay, but the delivery was delayed by a few days. 🚚⌛', 'rating': 3},
    {"review_text": "खूपच छान उत्पादन आहे, मला खूप आवडलं.", "rating": 5},
    {"review_text": "पैसे वाया गेले, अजिबात घेऊ नका.", "rating": 1},
    {"review_text": "मस्त आहे, दर्जा खूप चांगला आहे.", "rating": 5},
    {"review_text": "बकवास क्वालिटी, दोन दिवसात खराब झालं.", "rating": 1},
    {"review_text": "किंमत जास्त आहे पण प्रॉडक्ट चांगलं आहे.", "rating": 4},
    {"review_text": "नक्की खरेदी करा, खूप उपयोगी आहे.", "rating": 5},
    {"review_text": "सर्व्हिस खूप स्लो आहे, डिलिव्हरी उशिरा मिळाली.", "rating": 2},
    {"review_text": "रंग आणि डिझाइन भारी आहे, एकदम मस्त!", "rating": 5},
    {"review_text": "वापरण्यास सोपे आणि हलके आहे.", "rating": 4},
    {"review_text": "खराब पॅकेजिंग, बॉक्स तुटलेला होता.", "rating": 1},
    {"review_text": "या किमतीत यापेक्षा चांगलं काहीच मिळणार नाही.", "rating": 5},
    {"review_text": "साइज मध्ये प्रॉब्लेम आहे, खूप लहान आहे.", "rating": 2},
    {"review_text": "मला हे खूप आवडलं, धन्यवाद फ्लिपकार्ट!", "rating": 5},
    {"review_text": "टिकाऊ नाही, लगेच तुटलं.", "rating": 1},
    {"review_text": "नक्कीच शिफारस करेन, उत्तम दर्जा.", "rating": 5},
    {"review_text": "डिस्प्ले चांगला नाही, डोळ्यांना त्रास होतो.", "rating": 2},
    {"review_text": "बॅटरी बॅकअप खूप कमी आहे.", "rating": 2},
    {"review_text": "लुक खूप प्रीमियम आहे, आवडलं!", "rating": 5},
    {"review_text": "साधा प्रॉडक्ट आहे, काही खास नाही.", "rating": 3},
    {"review_text": "खूप फास्ट डिलिव्हरी, थँक्स!", "rating": 5},
    {"review_text": "फोटोमध्ये दिसतं तसं नाहीये.", "rating": 2},
    {"review_text": "आवडलं, कलर एकदम जबरदस्त आहे.", "rating": 5},
    {"review_text": "खूप महाग आहे, परवडत नाही.", "rating": 2},
    {"review_text": "मटेरियल खूप हलक्या प्रतीचे आहे.", "rating": 1},
    {"review_text": "चांगली खरेदी, मी समाधानी आहे.", "rating": 5},
    {"review_text": "बटण नीट काम करत नाहीत.", "rating": 2},
    {"review_text": "आवाज खूप स्पष्ट आहे, बेस्ट स्पीकर.", "rating": 5},
    {"review_text": "नेहमीप्रमाणे बेस्ट सर्व्हिस.", "rating": 5},
    {"review_text": "सॉफ्टवेअर अपडेट नंतर हँग होत आहे.", "rating": 2},
    {"review_text": "कम्फर्टेबल आहे, पाय दुखत नाहीत.", "rating": 5},
    {"review_text": "दुसरीकडे स्वस्त मिळतंय, इथे महाग आहे.", "rating": 2},
    {"review_text": "पॅकिंग खूप भारी होतं, सुरक्षित मिळालं.", "rating": 5},
    {"review_text": "काम नीट करत नाही, वेळ वाया गेला.", "rating": 1},
    {"review_text": "दिसायला सुंदर आहे, घरच्यांना आवडलं.", "rating": 5},
    {"review_text": "वजन खूप जास्त आहे, पकडायला त्रास होतो.", "rating": 2},
    {"review_text": "नेक्स्ट डे डिलिव्हरी मिळाली, अमेझिंग!", "rating": 5},
    {"review_text": "चार्जिंगला खूप वेळ लागतो.", "rating": 2},
    {"review_text": "सर्वोत्कृष्ट उत्पादन, ५ स्टार!", "rating": 5},
    {"review_text": "खूप ओरिजनल वाटतंय, क्वालिटी भारी आहे.", "rating": 5},
    {"review_text": "वॉरंटी कार्ड मिळालं नाही.", "rating": 2},
    {"review_text": "आवडलं नाही, परत केलं मी.", "rating": 1},
    {"review_text": "हेल्थ साठी खूप चांगलं आहे.", "rating": 5},
    {"review_text": "कव्हरेज चांगलं नाहीये शहरात सुद्धा.", "rating": 2},
    {"review_text": "एकदम झकास, भारी आहे!", "rating": 5},
    {"review_text": "डिस्काउंट मध्ये मिळालं, मस्त डील होती.", "rating": 5},
    {"review_text": "स्क्रॅचेस होते प्रॉडक्ट वर.", "rating": 2},
    {"review_text": "खूप दिवसांपासून हवं होतं, मिळालं!", "rating": 5},
    {"review_text": "स्पेस खूप कमी आहे आतमध्ये.", "rating": 2},
    {"review_text": "चांगलं आहे पण टिकेल का माहित नाही.", "rating": 3},
    {"review_text": "नक्कीच घ्या, डोळे झाकून विश्वास ठेवा.", "rating": 5},
    {'review_text': 'Not worth the price. The quality is poor and it broke after a week of use. 😞💔', 'rating': 2},
    {'review_text': 'Excellent product! The quality is top-notch and the delivery was fast. 😍👍🚚', 'rating': 5},
    {'review_text': 'Terrible experience! The item was defective and the customer service was unhelpful. 😡👎', 'rating': 1},
    {'review_text': 'The product is average, but the delivery was on time. 🚚⌛', 'rating': 3},
    {"review_text": "बहुत ही बढ़िया प्रोडक्ट है, जरूर खरीदें।", "rating": 5},
    {"review_text": "पैसे की बर्बादी है, मत लेना कोई भी।", "rating": 1},
    {"review_text": "क्वालिटी बहुत अच्छी है, मैं खुश हूँ।", "rating": 5},
    {"review_text": "बकवास है, दो दिन में टूट गया।", "rating": 1},
    {"review_text": "कीमत थोड़ी ज्यादा है पर काम अच्छा है।", "rating": 4},
    {"review_text": "बहुत उपयोगी है, डेली यूज के लिए बेस्ट।", "rating": 5},
    {"review_text": "डिलीवरी बहुत लेट हुई, सर्विस खराब है।", "rating": 2},
    {"review_text": "कलर और डिजाइन मस्त है, एकदम जबरदस्त!", "rating": 5},
    {"review_text": "इस्तेमाल करने में आसान और हल्का है।", "rating": 4},
    {"review_text": "पैकेजिंग खराब थी, बॉक्स फटा हुआ था।", "rating": 1},
    {"review_text": "इस बजट में इससे अच्छा कुछ नहीं मिलेगा।", "rating": 5},
    {"review_text": "साइज बहुत छोटा है, फिट नहीं हुआ।", "rating": 2},
    {"review_text": "मुझे यह बहुत पसंद आया, थैंक्स फ्लिपकार्ट!", "rating": 5},
    {"review_text": "मजबूत नहीं है, जल्दी खराब हो गया।", "rating": 1},
    {"review_text": "बिल्कुल रिकमेंड करूँगा, बेस्ट चॉइस।", "rating": 5},
    {"review_text": "डिस्प्ले अच्छा नहीं है, आँखों में चुभता है।", "rating": 2},
    {"review_text": "बैटरी बैकअप बहुत कम है इसका।", "rating": 2},
    {"review_text": "लुक बहुत प्रीमियम है, क्लासी लगता है।", "rating": 5},
    {"review_text": "साधारण सा प्रोडक्ट है, कुछ खास नहीं।", "rating": 3},
    {"review_text": "बहुत फास्ट डिलीवरी हुई, मजा आ गया!", "rating": 5},
    {"review_text": "फोटो जैसा दिख रहा था वैसा नहीं है।", "rating": 2},
    {"review_text": "कलर एकदम सही है जैसा मुझे चाहिए था।", "rating": 5},
    {"review_text": "बहुत महंगा है, इतना पैसा वसूल नहीं है।", "rating": 2},
    {"review_text": "मटेरियल बहुत लो क्वालिटी का है।", "rating": 1},
    {"review_text": "अच्छी डील है, मैं संतुष्ट हूँ।", "rating": 5},
    {"review_text": "बटन ठीक से काम नहीं कर रहे हैं।", "rating": 2},
    {"review_text": "आवाज बहुत साफ़ है, बेस्ट ब्लूटूथ स्पीकर।", "rating": 5},
    {"review_text": "हमेशा की तरह बेस्ट सर्विस और प्रोडक्ट।", "rating": 5},
    {"review_text": "सॉफ्टवेयर अपडेट के बाद स्लो हो गया।", "rating": 2},
    {"review_text": "बहुत कंफर्टेबल है, पहनकर अच्छा लगा।", "rating": 5},
    {"review_text": "दूसरी जगह सस्ता मिल रहा है, यहाँ नहीं।", "rating": 2},
    {"review_text": "पैकिंग बहुत अच्छी थी, सुरक्षित मिला।", "rating": 5},
    {"review_text": "काम ठीक से नहीं कर रहा, टाइम वेस्ट है।", "rating": 1},
    {"review_text": "दिखने में सुंदर है, घर वालों को पसंद आया।", "rating": 5},
    {"review_text": "वजन बहुत ज्यादा है, हाथ दर्द करने लगता है।", "rating": 2},
    {"review_text": "अगले ही दिन डिलीवरी मिल गई, शानदार!", "rating": 5},
    {"review_text": "चार्जिंग में बहुत टाइम लेता है यह फोन।", "rating": 2},
    {"review_text": "सर्वश्रेष्ठ प्रोडक्ट, ५ स्टार रेटिंग!", "rating": 5},
    {"review_text": "एकदम ओरिजिनल है, क्वालिटी बहुत भारी है।", "rating": 5},
    {"review_text": "वॉरंटी कार्ड साथ में नहीं मिला।", "rating": 2},
    {"review_text": "पसंद नहीं आया, मैंने वापस कर दिया।", "rating": 1},
    {"review_text": "हेल्थ के लिए बहुत फायदेमंद है यह।", "rating": 5},
    {"review_text": "नेटवर्क कवरेज अच्छा नहीं है घर के अंदर।", "rating": 2},
    {"review_text": "एकदम झकास, बहुत बढ़िया लगा!", "rating": 5},
    {"review_text": "डिस्काउंट में मिला, पैसा वसूल डील थी।", "rating": 5},
    {"review_text": "प्रोडक्ट पर खरोंच के निशान थे।", "rating": 2},
    {"review_text": "काफी समय से ढूंढ रहा था, फाइनली मिल गया!", "rating": 5},
    {"review_text": "अंदर काफी कम जगह है, छोटा है।", "rating": 2},
    {"review_text": "ठीक है पर लंबे समय तक चलेगा या नहीं पता नहीं।", "rating": 3},
    {"review_text": "आंख बंद करके खरीद लो, बेस्ट है।", "rating": 5},
    {'review_text': 'I am very satisfied with this purchase! The product is of great quality and the delivery was prompt. 😍👍🚚', 'rating': 5},
    {'review_text': 'The worst product I have ever bought! It stopped working after a day and the customer service was terrible. 😡👎', 'rating': 1},
    {'review_text': 'The product is okay, but the delivery was delayed by a few days. 🚚⌛', 'rating': 3},
    {'review_text': 'Not worth the price. The quality is poor and it broke after a week of use. 😞💔', 'rating': 2}
]

In [10]:
# create a DataFrame from the manual reviews and add necessary columns

df_manual = pd.DataFrame(manual_reviews)

# counts the number of words in the review text
df_manual['word_count'] = df_manual['review_text'].apply(lambda x: len(str(x).split()))

# sets fake and duplicate 0 because manually created
df_manual['is_fake'] = 0
df_manual['is_duplicate'] = 0

In [11]:
# merge the original dataset with the manual reviews
df_enriched = pd.concat([df_final, df_manual], ignore_index=True)

In [12]:
# emoji extraction function
# extract_actual_emojis => a function that takes a text input and returns a string containing only the emojis present in the text

def extract_actual_emojis(text):
    if not isinstance(text, str): # checks if the input text is not a string (e.g., it could be NaN or another data type), and if so, it returns an empty string to avoid errors during emoji extraction
        return "" 
    # emoji.is_emoji(c) => checks if the character c is an emoji, and if so, it includes it in the resulting string that contains only emojis from the original text
    return ''.join(c for c in text if emoji.is_emoji(c))

# applies the extract_actual_emojis function to the 'review_text' column of the df_enriched DataFrame, creating a new column 'emojis' that contains only the emojis extracted from the review text
df_enriched['emojis'] = df_enriched['review_text'].apply(extract_actual_emojis)

In [13]:
# sentiment categorization based on rating

def get_sentiment(rating):
    if rating >= 4:
        return 'Positive'
    elif rating <= 2:
        return 'Negative'
    else:
        return 'Neutral'

# applies the get_sentiment function to the 'rating' column of the df_enriched DataFrame, creating a new column 'sentiment' that categorizes each review as 'Positive', 'Negative', or 'Neutral' based on its rating
df_enriched['sentiment'] = df_enriched['rating'].apply(get_sentiment)

In [14]:
# shuffling the enriched DataFrame to ensure that the order of reviews is random, which can help prevent any bias during model training or analysis
df_enriched = df_enriched.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
# saving the enriched DataFrame to a new CSV file for further analysis or modeling, with UTF-8 encoding to handle any special characters or emojis properly
df_enriched.to_csv('../data/enriched_data.csv', index=False, encoding='utf-8-sig')

In [16]:
print("Total dataset size:", len(df_enriched))

Total dataset size: 1429


In [17]:
# display the last 5 rows to verify the enriched dataset
print("\nLast 5 rows (manual reviews):")
print(df_enriched[['review_text', 'emojis', 'rating']].tail())


Last 5 rows (manual reviews):
                                            review_text emojis  rating
1424                                          Awesome!!            5.0
1425  a good product but switch gets stuck most of t...            4.0
1426                         पैसे वाया गेले, घेऊ नका. 👎      👎     1.0
1427                  One of the best buy from FLIPKART            5.0
1428                       Really Nice, value for money            4.0
